In [ ]:
from google.colab import drive
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# 1. Drive'ı Bağla
drive.mount('/content/drive')

# 2. Veri Yükleme
dosya_yolu = '/content/drive/MyDrive/Adsız klasör/Telco_customer_churn.xlsx'
df = pd.read_excel(dosya_yolu)

print("Okunan Dosyadaki Toplam Satır Sayısı:", df.shape[0])

# 3. ÖNCE ÇÖPLERİ ATIYORUZ! (Hayat kurtaran değişiklik burada)
columns_to_drop = ['Churn Label', 'Churn Score', 'Churn Reason', 'CustomerID',
                   'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']

df = df.drop(columns=columns_to_drop, errors='ignore')

# 4. Total Charges İşlemi
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

# 5. ŞİMDİ BOŞLUKLARI SİLEBİLİRİZ (Artık sadece faturası boş olan 11 satır silinecek)
df.dropna(inplace=True)

# 6. X ve y'yi Ayırma
X = df.drop(columns=['Churn Value'], errors='ignore')
y = df['Churn Value']

# 7. Veriyi Bölme (stratify=y ile mükemmel dağılım)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# 8. Sayısallaştırma (Encoding) ve Hizalama
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_val_encoded = pd.get_dummies(X_val, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

X_val_encoded, _ = X_val_encoded.align(X_train_encoded, join='right', axis=1, fill_value=0)
X_test_encoded, _ = X_test_encoded.align(X_train_encoded, join='right', axis=1, fill_value=0)

# 9. SMOTE İŞLEMİ (Dengeleme)
print("\n--- SMOTE ÖNCESİ Sınıf Dağılımı (y_train) ---")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_encoded, y_train)

print("\n--- SMOTE SONRASI Sınıf Dağılımı (y_train_smote) ---")
print(y_train_smote.value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Okunan Dosyadaki Toplam Satır Sayısı: 7043

--- SMOTE ÖNCESİ Sınıf Dağılımı (y_train) ---
Churn Value
0    4130
1    1495
Name: count, dtype: int64

--- SMOTE SONRASI Sınıf Dağılımı (y_train_smote) ---
Churn Value
0    4130
1    4130
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Modeli Çağıralım (Sonuçların tutarlı olması için random_state=42 kullanıyoruz)
model = RandomForestClassifier(random_state=42)

# 2. Modeli EĞİTİYORUZ! (Sadece SMOTE ile dengelenmiş verileri veriyoruz)
print("Model eğitiliyor, lütfen bekleyin...")
model.fit(X_train_smote, y_train_smote)
print("Eğitim tamamlandı!\n")

# 3. Validasyon Seti Üzerinde Sınav Vakti
# (Modele X_val sorularını verip tahminlerini alıyoruz)
y_val_pred = model.predict(X_val_encoded)

# 4. Sonuç Karnesi (Değerlendirme Raporu)
print("--- VALIDASYON (ARA SINAV) SONUÇLARI ---")
print(classification_report(y_val, y_val_pred))

print("\n--- KARMAŞIKLIK MATRİSİ (Confusion Matrix) ---")
print(confusion_matrix(y_val, y_val_pred))

Model eğitiliyor, lütfen bekleyin...
Eğitim tamamlandı!

--- VALIDASYON (ARA SINAV) SONUÇLARI ---
              precision    recall  f1-score   support

           0       0.85      0.84      0.85       516
           1       0.58      0.59      0.59       187

    accuracy                           0.78       703
   macro avg       0.71      0.72      0.72       703
weighted avg       0.78      0.78      0.78       703


--- KARMAŞIKLIK MATRİSİ (Confusion Matrix) ---
[[435  81]
 [ 76 111]]
